In [1]:
from google.colab import drive
import os

# Mount the drive to ensure connection
drive.mount('/content/drive')



Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
import os

# Define the dataset path
data_dir = '/content/drive/MyDrive/AgriScan_Dataset'

phases = ['train', 'val']
classes = ['Potato', 'Tomato', 'Background']

print("Dataset Image Count:")
print("-" * 20)

for phase in phases:
    phase_dir = os.path.join(data_dir, phase)
    total_phase = 0
    print(f"Phase: {phase.upper()}")

    for cls in classes:
        cls_dir = os.path.join(phase_dir, cls)
        if os.path.exists(cls_dir):
            count = len(os.listdir(cls_dir))
            total_phase += count
            print(f"  - {cls}: {count} images")
        else:
            print(f"  - {cls}: Directory not found!")

    print(f"  Total in {phase}: {total_phase}\n")

Dataset Image Count:
--------------------
Phase: TRAIN
  - Potato: 1000 images
  - Tomato: 1000 images
  - Background: 1000 images
  Total in train: 3000

Phase: VAL
  - Potato: 250 images
  - Tomato: 250 images
  - Background: 250 images
  Total in val: 750



In [3]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, models, transforms
from torch.utils.data import DataLoader

# Device configuration
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Data augmentation and normalization
data_transforms = {
    'train': transforms.Compose([
        transforms.RandomResizedCrop(224),
        transforms.RandomHorizontalFlip(),
        transforms.RandomVerticalFlip(),
        transforms.ColorJitter(brightness=0.2, contrast=0.2),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ]),
    'val': transforms.Compose([
        transforms.Resize(256),
        transforms.CenterCrop(224),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ]),
}

# Create datasets and dataloaders
image_datasets = {x: datasets.ImageFolder(os.path.join(data_dir, x), data_transforms[x])
                  for x in ['train', 'val']}

dataloaders = {x: DataLoader(image_datasets[x], batch_size=32, shuffle=True, num_workers=2)
               for x in ['train', 'val']}

dataset_sizes = {x: len(image_datasets[x]) for x in ['train', 'val']}
class_names = image_datasets['train'].classes

# Load pretrained ResNet18 model
model = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)

# FREEZE EARLY LAYERS: This makes training much faster and protects pretrained weights
for param in model.parameters():
    param.requires_grad = False

# Modify the final classification layer for 3 classes
num_ftrs = model.fc.in_features
model.fc = nn.Linear(num_ftrs, len(class_names))

model = model.to(device)

# Define loss function and optimizer
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

print(f"Classes established: {class_names}")
print("Model, criterion, and optimizer loaded successfully.")

Using device: cuda:0
Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 128MB/s]


Classes established: ['Background', 'Potato', 'Tomato']
Model, criterion, and optimizer loaded successfully.


In [4]:
import time
import copy

def train_model(model, criterion, optimizer, num_epochs=5):
    since = time.time()

    # Store the best model weights
    best_model_wts = copy.deepcopy(model.state_dict())
    best_acc = 0.0

    for epoch in range(num_epochs):
        print(f'Epoch {epoch}/{num_epochs - 1}')
        print('-' * 10)

        # Each epoch has a training and validation phase
        for phase in ['train', 'val']:
            if phase == 'train':
                model.train()  # Set model to training mode
            else:
                model.eval()   # Set model to evaluate mode

            running_loss = 0.0
            running_corrects = 0

            # Iterate over data batches
            for inputs, labels in dataloaders[phase]:
                inputs = inputs.to(device)
                labels = labels.to(device)

                # Zero the parameter gradients
                optimizer.zero_grad()

                # Forward pass
                with torch.set_grad_enabled(phase == 'train'):
                    outputs = model(inputs)
                    _, preds = torch.max(outputs, 1)
                    loss = criterion(outputs, labels)

                    # Backward pass and optimize only in training phase
                    if phase == 'train':
                        loss.backward()
                        optimizer.step()

                # Calculate statistics
                running_loss += loss.item() * inputs.size(0)
                running_corrects += torch.sum(preds == labels.data)

            # Calculate epoch loss and accuracy
            epoch_loss = running_loss / dataset_sizes[phase]
            epoch_acc = running_corrects.double() / dataset_sizes[phase]

            print(f'{phase.capitalize()} Loss: {epoch_loss:.4f} Acc: {epoch_acc:.4f}')

            # Deep copy the model if it achieves the best validation accuracy
            if phase == 'val' and epoch_acc > best_acc:
                best_acc = epoch_acc
                best_model_wts = copy.deepcopy(model.state_dict())

        print()

    time_elapsed = time.time() - since
    print(f'Training complete in {time_elapsed // 60:.0f}m {time_elapsed % 60:.0f}s')
    print(f'Best val Acc: {best_acc:.4f}')

    # Load the best model weights before returning
    model.load_state_dict(best_model_wts)
    return model

# Execute the training function
model_ft = train_model(model, criterion, optimizer, num_epochs=5)

Epoch 0/4
----------
Train Loss: 0.5976 Acc: 0.7767
Val Loss: 0.3015 Acc: 0.9133

Epoch 1/4
----------
Train Loss: 0.3384 Acc: 0.8843
Val Loss: 0.2572 Acc: 0.9160

Epoch 2/4
----------
Train Loss: 0.2684 Acc: 0.9127
Val Loss: 0.1875 Acc: 0.9467

Epoch 3/4
----------
Train Loss: 0.2599 Acc: 0.9087
Val Loss: 0.2352 Acc: 0.9093

Epoch 4/4
----------
Train Loss: 0.2425 Acc: 0.9083
Val Loss: 0.1579 Acc: 0.9520

Training complete in 9m 8s
Best val Acc: 0.9520


In [5]:
import torch

# Define path and save the trained model to Google Drive
save_path = '/content/drive/MyDrive/AgriScan_Dataset/agriscan_model.pth'
torch.save(model_ft.state_dict(), save_path)
print(f"Model successfully saved to: {save_path}")

Model successfully saved to: /content/drive/MyDrive/AgriScan_Dataset/agriscan_model.pth
